<a href="https://colab.research.google.com/github/BandanaSingha24/02_Genomics-end-to-end-Pipelines-for-Bioinformatics/blob/main/Somatic_Variant_Analysis_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install tools (Conda is slow -> use apt + prebuilt binaries where possible)
!apt-get update && apt-get install -y samtools bwa fastqc tabix
!pip install cyvcf2 # for VCF reading later

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [95.6 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates/multiverse amd64 Packages [86.4 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,294 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,355 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,603 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease


In [2]:
# Install GATK (overwrite automatically)
!wget -q https://github.com/broadinstitute/gatk/releases/download/4.5.0.0/gatk-4.5.0.0.zip
!unzip -o -q gatk-4.5.0.0.zip
!mv gatk-4.5.0.0/gatk /usr/local/bin/
!chmod +x /usr/local/bin/gatk
print("GATK installed successfully!")

GATK installed successfully!


In [3]:
# Step 1: Download tumor-normal FASTQ files
import os
os.makedirs("data/fastq", exist_ok=True)

base = "https://zenodo.org/record/2582555/files/"
files = [
    "SLGFSK-N_231335_r1_chr5_12_17.fastq.gz",
    "SLGFSK-N_231335_r2_chr5_12_17.fastq.gz",
    "SLGFSK-T_231336_r1_chr5_12_17.fastq.gz",
    "SLGFSK-T_231336_r2_chr5_12_17.fastq.gz"
]
for f in files:
    !wget -nc {base + f} -O data/fastq/{f}

print("FASTQ files downloaded successfully!")
!ls -lh data/fastq/

--2026-05-22 18:34:22--  https://zenodo.org/record/2582555/files/SLGFSK-N_231335_r1_chr5_12_17.fastq.gz
Resolving zenodo.org (zenodo.org)... 188.185.48.75, 188.184.103.118, 137.138.52.235, ...
Connecting to zenodo.org (zenodo.org)|188.185.48.75|:443... connected.
HTTP request sent, awaiting response... 301 MOVED PERMANENTLY
Location: /records/2582555/files/SLGFSK-N_231335_r1_chr5_12_17.fastq.gz [following]
--2026-05-22 18:34:22--  https://zenodo.org/records/2582555/files/SLGFSK-N_231335_r1_chr5_12_17.fastq.gz
Reusing existing connection to zenodo.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 530427108 (506M) [application/octet-stream]
Saving to: ‘data/fastq/SLGFSK-N_231335_r1_chr5_12_17.fastq.gz’

data/fastq/SLGFSK-N 100%[===================>] 505.85M  12.1MB/s    in 44s     

2026-05-22 18:35:06 (11.5 MB/s) - ‘data/fastq/SLGFSK-N_231335_r1_chr5_12_17.fastq.gz’ saved [530427108/530427108]

--2026-05-22 18:35:06--  https://zenodo.org/record/2582555/files/SLGFSK-N_23133

In [4]:
# Step 2: FASTQ QC
# FASTQC - Quality Control
!mkdir -p results/fastqc
!fastqc -o results/fastqc -t 2 data/fastq/*.fastq.gz

Started analysis of SLGFSK-N_231335_r1_chr5_12_17.fastq.gz
Started analysis of SLGFSK-N_231335_r2_chr5_12_17.fastq.gz
Approx 5% complete for SLGFSK-N_231335_r2_chr5_12_17.fastq.gz
Approx 5% complete for SLGFSK-N_231335_r1_chr5_12_17.fastq.gz
Approx 10% complete for SLGFSK-N_231335_r2_chr5_12_17.fastq.gz
Approx 10% complete for SLGFSK-N_231335_r1_chr5_12_17.fastq.gz
Approx 15% complete for SLGFSK-N_231335_r2_chr5_12_17.fastq.gz
Approx 15% complete for SLGFSK-N_231335_r1_chr5_12_17.fastq.gz
Approx 20% complete for SLGFSK-N_231335_r2_chr5_12_17.fastq.gz
Approx 20% complete for SLGFSK-N_231335_r1_chr5_12_17.fastq.gz
Approx 25% complete for SLGFSK-N_231335_r2_chr5_12_17.fastq.gz
Approx 25% complete for SLGFSK-N_231335_r1_chr5_12_17.fastq.gz
Approx 30% complete for SLGFSK-N_231335_r2_chr5_12_17.fastq.gz
Approx 30% complete for SLGFSK-N_231335_r1_chr5_12_17.fastq.gz
Approx 35% complete for SLGFSK-N_231335_r2_chr5_12_17.fastq.gz
Approx 35% complete for SLGFSK-N_231335_r1_chr5_12_17.fastq.gz
Ap

In [5]:
# Step 3: Alignment → BAM with BWA-MEM

# FAST Subsampling + Quick Alignment
import os
os.makedirs("data/subsampled_fastq", exist_ok=True)
os.makedirs("results", exist_ok=True)

# 1. Download the pre-sliced reference (hg19 chr5_12_17 Mb slice)
!wget -nc https://zenodo.org/record/2582555/files/hg19.chr5_12_17.fa.gz
!gunzip -f hg19.chr5_12_17.fa.gz
!mv hg19.chr5_12_17.fa ref_chr5_slice.fa

# Index reference (very fast)
!bwa index ref_chr5_slice.fa
!samtools faidx ref_chr5_slice.fa

# 2. Subsample to first 100,000 read pairs per file
import os
os.makedirs("data/subsampled_fastq", exist_ok=True)
os.makedirs("results", exist_ok=True)

number_of_pairs = 50000
lines_to_take = number_of_pairs * 4

samples = ['N', 'T'] # N=normal, T=tumor
for sample in samples:
    for read in [1, 2]:
        input_file = f"data/fastq/SLGFSK-{sample}_23133{3 if sample=='T' else 2}_r{read}_chr5_12_17.fastq.gz"
        output_file = f"data/subsampled_fastq/SLGFSK-{sample}_sub_r{read}.fastq.gz"
        !zcat {input_file} | head -n {lines_to_take} | gzip > {output_file}

!ls -lh data/subsampled_fastq/

# 3. Align subsampled normal
!bwa mem -t 4 -R '@RG\tID:normal\tSM:normal\tLB:lib' ref_chr5_slice.fa \
    data/subsampled_fastq/SLGFSK-N_sub_r1.fastq.gz \
    data/subsampled_fastq/SLGFSK-N_sub_r2.fastq.gz | \
    samtools sort -o results/normal.bam -
!samtools index results/normal.bam

# Align subsampled tumor
!bwa mem -t 4 -R '@RG\tID:tumor\tSM:tumor\tLB:lib' ref_chr5_slice.fa \
    data/subsampled_fastq/SLGFSK-T_sub_r1.fastq.gz \
    data/subsampled_fastq/SLGFSK-T_sub_r2.fastq.gz | \
    samtools sort -o results/tumor.bam -
!samtools index results/tumor.bam

print("Subsampled BAM files ready!")
!ls -lh results/*.bam
!du -sh results/*.bam # shows human-readable sizes

--2026-05-22 18:51:54--  https://zenodo.org/record/2582555/files/hg19.chr5_12_17.fa.gz
Resolving zenodo.org (zenodo.org)... 137.138.52.235, 188.184.103.118, 188.184.98.114, ...
Connecting to zenodo.org (zenodo.org)|137.138.52.235|:443... connected.
HTTP request sent, awaiting response... 301 MOVED PERMANENTLY
Location: /records/2582555/files/hg19.chr5_12_17.fa.gz [following]
--2026-05-22 18:51:55--  https://zenodo.org/records/2582555/files/hg19.chr5_12_17.fa.gz
Reusing existing connection to zenodo.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 126162910 (120M) [application/octet-stream]
Saving to: ‘hg19.chr5_12_17.fa.gz’

hg19.chr5_12_17.fa. 100%[===================>] 120.32M  19.5MB/s    in 7.4s    

2026-05-22 18:52:02 (16.3 MB/s) - ‘hg19.chr5_12_17.fa.gz’ saved [126162910/126162910]

[bwa_index] Pack FASTA... 4.71 sec
[bwa_index] Construct BWT for the packed sequence...
[BWTIncCreate] textLength=791924730, availableWord=67722660
[BWTIncConstructFromPacked] 10 itera

In [8]:
# Step 4: Somatic Variant Calling with Mutect2

# Ensure reference is fully prepared for GATK

import os

gatk_jar = None
# Is baar hum sirf 'local.jar' wali file dhoondhenge
for root, dirs, files in os.walk("/content"):
    for file in files:
        if "gatk-package" in file and "local.jar" in file:
            gatk_jar = os.path.join(root, file)
            break
    if gatk_jar: break

if gatk_jar:
    print(f"Sahi GATK jar mil gayi: {gatk_jar}")
    !java -jar {gatk_jar} CreateSequenceDictionary -R ref_chr5_slice.fa
else:
    print("GATK local jar nahi mili. Check karein ki installation sahi hua hai.")

Sahi GATK jar mil gayi: /content/gatk-4.5.0.0/gatk-package-4.5.0.0-local.jar
INFO	2026-05-22 19:12:22	CreateSequenceDictionary	Output dictionary will be written in ref_chr5_slice.dict
19:12:22.933 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/gatk-4.5.0.0/gatk-package-4.5.0.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
[Fri May 22 19:12:22 UTC 2026] CreateSequenceDictionary --REFERENCE ref_chr5_slice.fa --TRUNCATE_NAMES_AT_WHITESPACE true --NUM_SEQUENCES 2147483647 --VERBOSITY INFO --QUIET false --VALIDATION_STRINGENCY STRICT --COMPRESSION_LEVEL 2 --MAX_RECORDS_IN_RAM 500000 --CREATE_INDEX false --CREATE_MD5_FILE false --help false --version false --showHidden false --USE_JDK_DEFLATER false --USE_JDK_INFLATER false
[Fri May 22 19:12:23 UTC 2026] Executing as root@a55219240c30 on Linux 6.6.122+ amd64; OpenJDK 64-Bit Server VM 17.0.18+8-Ubuntu-122.04.1; Deflater: Intel; Inflater: Intel; Provider GCS is available; Picard version: Version:4.5.0.

In [9]:
# Mutect2 Variant Calling
!java -jar {gatk_jar} Mutect2 -R ref_chr5_slice.fa -I results/tumor.bam -I results/normal.bam -normal normal -O result.vcf.gz

19:17:11.953 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/gatk-4.5.0.0/gatk-package-4.5.0.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
19:17:12.217 INFO  Mutect2 - ------------------------------------------------------------
19:17:12.225 INFO  Mutect2 - The Genome Analysis Toolkit (GATK) v4.5.0.0
19:17:12.225 INFO  Mutect2 - For support and documentation go to https://software.broadinstitute.org/gatk/
19:17:12.225 INFO  Mutect2 - Executing as root@a55219240c30 on Linux v6.6.122+ amd64
19:17:12.226 INFO  Mutect2 - Java runtime: OpenJDK 64-Bit Server VM v17.0.18+8-Ubuntu-122.04.1
19:17:12.226 INFO  Mutect2 - Start Date/Time: May 22, 2026 at 7:17:11 PM UTC
19:17:12.226 INFO  Mutect2 - ------------------------------------------------------------
19:17:12.226 INFO  Mutect2 - ------------------------------------------------------------
19:17:12.229 INFO  Mutect2 - HTSJDK Version: 4.1.0
19:17:12.231 INFO  Mutect2 - Picard Version: 3.1.1
19:17:12.

In [10]:
!ls -lh result.vcf.gz

-rw-r--r-- 1 root root 4.2K May 22 19:18 result.vcf.gz


In [11]:
!zcat result.vcf.gz|head -n20

##fileformat=VCFv4.2
##FORMAT=<ID=AD,Number=R,Type=Integer,Description="Allelic depths for the ref and alt alleles in the order listed">
##FORMAT=<ID=AF,Number=A,Type=Float,Description="Allele fractions of alternate alleles in the tumor">
##FORMAT=<ID=DP,Number=1,Type=Integer,Description="Approximate read depth (reads with MQ=255 or with bad mates are filtered)">
##FORMAT=<ID=F1R2,Number=R,Type=Integer,Description="Count of reads in F1R2 pair orientation supporting each allele">
##FORMAT=<ID=F2R1,Number=R,Type=Integer,Description="Count of reads in F2R1 pair orientation supporting each allele">
##FORMAT=<ID=FAD,Number=R,Type=Integer,Description="Count of fragments supporting each allele.">
##FORMAT=<ID=GQ,Number=1,Type=Integer,Description="Genotype Quality">
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
##FORMAT=<ID=PGT,Number=1,Type=String,Description="Physical phasing haplotype information, describing how the alternate alleles are phased in relation to one another; wil

In [12]:
# Step 8: Filter Mutect2 Results

!java -jar {gatk_jar} FilterMutectCalls \
 -R ref_chr5_slice.fa \
 -V result.vcf.gz \
 -O filtered_somatic.vcf.gz

19:40:01.554 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/gatk-4.5.0.0/gatk-package-4.5.0.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
19:40:02.063 INFO  FilterMutectCalls - ------------------------------------------------------------
19:40:02.073 INFO  FilterMutectCalls - The Genome Analysis Toolkit (GATK) v4.5.0.0
19:40:02.074 INFO  FilterMutectCalls - For support and documentation go to https://software.broadinstitute.org/gatk/
19:40:02.074 INFO  FilterMutectCalls - Executing as root@a55219240c30 on Linux v6.6.122+ amd64
19:40:02.074 INFO  FilterMutectCalls - Java runtime: OpenJDK 64-Bit Server VM v17.0.18+8-Ubuntu-122.04.1
19:40:02.077 INFO  FilterMutectCalls - Start Date/Time: May 22, 2026 at 7:40:01 PM UTC
19:40:02.078 INFO  FilterMutectCalls - ------------------------------------------------------------
19:40:02.078 INFO  FilterMutectCalls - ------------------------------------------------------------
19:40:02.080 INFO  FilterMutect

In [14]:
# visulation

# 1. Variant Calling (Mutect2)
!java -jar {gatk_jar} Mutect2 \
 -R ref_chr5_slice.fa \
 -I normal.bam \
 -I tumor.bam \
 -O result.vcf.gz

# 2. Filtering (FilterMutectCalls)
!java -jar {gatk_jar} FilterMutectCalls \
 -R ref_chr5_slice.fa \
 -V result.vcf.gz \
 -O filtered_somatic.vcf.gz

# 3. Visualization (Table ke roop mein dekhne ke liye)
import pandas as pd
!gunzip -c filtered_somatic.vcf.gz > filtered_somatic.vcf
df = pd.read_csv('filtered_somatic.vcf', sep='\t', comment='#', names=['CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO', 'FORMAT', 'SAMPLE'])
print(df[['CHROM', 'POS', 'REF', 'ALT', 'FILTER']].head(10))

19:56:44.251 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/gatk-4.5.0.0/gatk-package-4.5.0.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
19:56:44.522 INFO  Mutect2 - ------------------------------------------------------------
19:56:44.527 INFO  Mutect2 - The Genome Analysis Toolkit (GATK) v4.5.0.0
19:56:44.527 INFO  Mutect2 - For support and documentation go to https://software.broadinstitute.org/gatk/
19:56:44.527 INFO  Mutect2 - Executing as root@a55219240c30 on Linux v6.6.122+ amd64
19:56:44.527 INFO  Mutect2 - Java runtime: OpenJDK 64-Bit Server VM v17.0.18+8-Ubuntu-122.04.1
19:56:44.528 INFO  Mutect2 - Start Date/Time: May 22, 2026 at 7:56:44 PM UTC
19:56:44.528 INFO  Mutect2 - ------------------------------------------------------------
19:56:44.528 INFO  Mutect2 - ------------------------------------------------------------
19:56:44.531 INFO  Mutect2 - HTSJDK Version: 4.1.0
19:56:44.531 INFO  Mutect2 - Picard Version: 3.1.1
19:56:44.

In [15]:
# Annotation  VEP/Funcotation
import pandas as pd

!grep -v "^#" filtered_somatic.vcf > variants_table.tsv

df = pd.read_csv('variants_table.tsv', sep='\t', names=['CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO', 'FORMAT', 'SAMPLE'])

print(df[['CHROM', 'POS', 'REF', 'ALT', 'FILTER']].head())

df.to_csv('final_annotated_variants.csv', index=False)

Empty DataFrame
Columns: [CHROM, POS, REF, ALT, FILTER]
Index: []
